# Checkout support: Lightning + GPT Sol 6.1

**Selected workshop demo.** “Checkout says payment failed, but my card was charged. Should I pay again?”

Lightning extracts backend facts, GPT reads the actual screenshot, and Lightning reconciles the support decision. Native Switchyard dispatches requests using authored routing rules.

Run all cells with the defaults below to replay saved real model responses. **No API key or model calls are needed in recorded mode.**

In [ ]:
from pathlib import Path
import sys, json, hashlib, statistics
import pandas as pd
from IPython.display import display, Image, Markdown

_start = Path.cwd().resolve()
ROOT = next(p for p in [_start, *_start.parents] if (p / 'shared/incident_demo.py').is_file())
EXAMPLE = ROOT / 'examples/checkout-support/lightning-gpt-sol'
CHECKOUT = EXAMPLE.parent
sys.path.insert(0, str(ROOT / 'shared'))
sys.path.insert(0, str(CHECKOUT))

In [ ]:
DEMO_MODE = "recorded"  # recorded | live
ALLOW_PAID_CALLS = False
SELECTED_REPEAT = 1  # 1, 2 or 3
REPEATS = 3  # only applies to a fresh live run
RECORDING = ROOT / 'recordings/checkout-support/lightning-gpt-sol-final-2026-10-17-20261017T025833913064Z.json'
assert DEMO_MODE in {'recorded', 'live'}
assert SELECTED_REPEAT in {1, 2, 3}

## The customer's incident

The screen and payment backend disagree. A duplicate webhook must not be counted as a second financial capture. No payment, refund or repair is performed by this demo.

In [ ]:
case = json.loads((CHECKOUT / 'case.json').read_text())
display(Markdown('> ' + case['ticket']))
display(Image(filename=str(CHECKOUT / 'checkout.png'), width=840))
display(pd.DataFrame([
    {'Backend evidence': 'Settled capture', 'Observed value': 'cap-42 · SGD 118.70 · one unique capture'},
    {'Backend evidence': 'Order state', 'Observed value': case['payment']['order_state']},
    {'Backend evidence': 'Repeated webhook', 'Observed value': 'Same capture ID; not another charge'},
    {'Backend evidence': 'Confirmation request', 'Observed value': 'HTTP 504'},
    {'Backend evidence': 'Runbook', 'Observed value': case['runbook']['id']},
]))

## Load the recording—or explicitly opt into fresh calls

Recorded mode reads all nine saved attempts, including the failed Lightning-only workflows. Live mode uses the same workflow and TOML profile through your own configured gateway. Changing both controls above to `DEMO_MODE = "live"` and `ALLOW_PAID_CALLS = True` enables paid calls; otherwise no inference is sent.

Fresh results are written under ignored `runs/checkout-support/`. The pinned workshop recording is preserved. Tracing is not enabled by this workflow.

In [ ]:
if DEMO_MODE == 'live':
    if not ALLOW_PAID_CALLS:
        raise ValueError('Live inference requires explicit ALLOW_PAID_CALLS = True.')
    from dotenv import load_dotenv
    import workflow
    load_dotenv(ROOT / '.env', override=True)
    _, active_recording = await workflow.compare(REPEATS, phase='lightning-gpt-notebook', profile_dir=EXAMPLE)
else:
    active_recording = RECORDING

recording = json.loads(active_recording.read_text())
results = recording['results']
rates = recording['metadata']['rates']['rates']
if DEMO_MODE == 'recorded':
    source_map = json.loads((ROOT / 'evidence/source-map.json').read_text())
    for original_path, digest in recording['metadata']['source_hashes'].items():
        measured = ROOT / source_map[original_path][digest]
        assert hashlib.sha256(measured.read_bytes()).hexdigest() == digest
assert all(r['cost_complete'] for r in results), 'Missing usage prevents a complete cost comparison.'
print(f'{DEMO_MODE.upper()} · {len(results)} task attempts · no hidden retries or excluded failures')

## Compare all three approaches

Costs include every recorded call, with reported cache reads and writes priced separately. Lightning-only costs cover **partial failed workflows**, not a completed solution. The fixed workflow allows the same two-attempt limit per stage in every arm.

In [ ]:
LABELS = {'super_only': 'GPT Sol 6.1 only', 'lightning_only': 'Lightning only', 'routed': 'Lightning + GPT via Switchyard'}

def ordinary_input_cost(result):
    return sum((c['usage']['input_tokens'] * rates[c['model']]['input_per_million']
                + c['usage']['output_tokens'] * rates[c['model']]['output_per_million']) / 1_000_000
               for c in result['calls'])

rows = []
for arm, label in LABELS.items():
    attempts = [r for r in results if r['arm'] == arm]
    accepted = sum(r['passed'] for r in attempts)
    total = sum(r['cost_usd'] for r in attempts)
    rows.append({'Approach': label, 'Accepted': f'{accepted}/{len(attempts)}',
                 'Mean estimated USD/workflow': total / len(attempts),
                 'USD/accepted including failures': total / accepted if accepted else None,
                 'Median seconds': statistics.median(r['elapsed_seconds'] for r in attempts),
                 'Model calls': sum(len(r['calls']) for r in attempts),
                 'Ordinary-input repricing USD/workflow': statistics.mean(ordinary_input_cost(r) for r in attempts)})
summary = pd.DataFrame(rows)
display(summary.style.format({'Mean estimated USD/workflow': '${:.6f}',
    'USD/accepted including failures': '${:.6f}', 'Median seconds': '{:.2f}',
    'Ordinary-input repricing USD/workflow': '${:.6f}'}, na_rep='No accepted results'))
base, routed = rows[0], rows[2]
saving = 1 - routed['Mean estimated USD/workflow'] / base['Mean estimated USD/workflow']
ordinary_saving = 1 - routed['Ordinary-input repricing USD/workflow'] / base['Ordinary-input repricing USD/workflow']
display(Markdown(f'**{saving:.1%} lower estimated spend** versus GPT everywhere with reported caching. '
                 f'**{ordinary_saving:.1%} lower** when all input is repriced at ordinary rates.'))

## Inspect a real routed attempt

Change `SELECTED_REPEAT` in the controls and rerun this cell to inspect repeats 1–3. The route below is drawn from actual call records, not a scripted model choice.

In [ ]:
selected = next(r for r in results if r['arm'] == 'routed' and r['repeat'] == SELECTED_REPEAT)
steps = []
for c in selected['calls']:
    steps.append({'Step': len(steps) + 1, 'Task': c['kind'],
        'Model': c['model'].rsplit('/', 1)[-1], 'Routing reason': c['decision_reason'],
        'Image sent': c['has_image'], 'Estimated USD': c['cost_usd'],
        'Seconds': c['seconds']})
display(pd.DataFrame(steps).style.format({'Estimated USD': '${:.6f}', 'Seconds': '{:.2f}'}))
display(Markdown(f"**Repeat {SELECTED_REPEAT}:** {'accepted' if selected['passed'] else 'not accepted'} · "
                 f"${selected['cost_usd']:.6f} · {selected['elapsed_seconds']:.2f} seconds"))
display(Markdown('### Proposed customer reply\n\n' + (selected['customer_reply'] or 'No accepted reply.')))
display(pd.Series(selected['report'], name='Validated support report').to_frame())

## Explain why the baselines differ

Lightning is text-only, so the Lightning-only baseline is blocked before sending the image. This is a modality limitation, not an observed failed image inference or a general reasoning failure. The routed workflow sends that actual image to GPT and keeps routine text on Lightning.

Switchyard uses `HeaderRoutingConfig` for dispatch. Application policy tags actual images and validator-rejected retries as capable; other text is efficient. There is no paid LLM judge or learned difficulty classifier. The three recorded routed attempts needed no retries.

In [ ]:
partial = []
for r in results:
    if r['arm'] == 'lightning_only':
        partial.append({'Repeat': r['repeat'], 'Accepted': r['passed'],
            'Stop reason': r['error_type'],
            'Detail': '; '.join(b['reason'] for b in r['blocked_requests']),
            'Partial estimated USD': r['cost_usd']})
display(pd.DataFrame(partial).sort_values('Repeat'))
price_rows = [{'Model': model.rsplit('/', 1)[-1], **price} for model, price in rates.items()]
display(pd.DataFrame(price_rows).set_index('Model').style.format('${:.2f}'))

## What to say in the workshop

“On this synthetic support workflow, Lightning handled the text and GPT read the screenshot. Both routed and GPT-only completed all three attempts. Routing reduced estimated spend by **85.7% with reported caching**, or **72.3% with ordinary-input repricing**.”

The pin uses OpenRouter's historical reference-price snapshot applied to gateway token usage, not actual invoices. A separate vision probe warmed the shared image prompt; repeat caches can carry over. Ordinary-input repricing removes cache discounts and write premiums from the observed counts; it is not a second live run.

This is one synthetic case repeated three times in a **fixed three-stage workflow**, not an autonomous-agent or general reliability benchmark. Validators use the labelled fixture, and the app renders the customer reply from validated integer cents.

[Full findings and evidence](README.md) · [HTML presentation fallback](preview.html) · [Model profile](models.toml) · [Workshop guide](../../../README.md)